# 01 – Time Series Clustering på NetCDF Cubes

**Formål:** Kør Time Series Clustering (TSC) på de 138 NetCDF space-time cubes fra `00_space_time_cube_processing.ipynb`.  
For hvert af de 23 variable sammenkædes alle tidsintervaller til en 30-årig profil per lokalitet, hvorefter `TimeSeriesKMeans` (DTW, 6 klynger) køres.  

**Outputs:**
- `data/processed/tsc_netcdf_vector/{var}_tsc.gpkg` – ét GeoPackage per variabel med kolonne `cluster_tsc` (0–5)
- `results/metrics/charts_html/{var}_tsc_timeseries.html` – interaktivt Plotly-tidsseriekort
- `data/processed/tsc_netcdf_summary.csv` – opsummerende statistik

In [29]:
# ── Cell 1: Imports & konfiguration ────────────────────────────────────────────
import os
import re
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import plotly.graph_objects as go
from sklearn.metrics import silhouette_score
from tslearn.clustering import TimeSeriesKMeans
from tslearn.preprocessing import TimeSeriesScalerMeanVariance

warnings.filterwarnings('ignore')

# ── Paths ───────────────────────────────────────────────────────────────────────
REPO_ROOT   = Path('..').resolve()
NETCDF_DIR  = REPO_ROOT / 'data' / 'processed' / 'NetCDF_cubes'
VECTOR_DIR  = REPO_ROOT / 'data' / 'processed' / 'tsc_netcdf_vector'
CHARTS_DIR  = REPO_ROOT / 'results' / 'metrics' / 'charts_html'
SHAPEFILE   = REPO_ROOT / 'data' / 'raw' / 'GIS_lag' / 'clusters_hovedstad_clean.shp'
SUMMARY_CSV = REPO_ROOT / 'data' / 'processed' / 'tsc_netcdf_summary.csv'

VECTOR_DIR.mkdir(parents=True, exist_ok=True)
CHARTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Clustering config ───────────────────────────────────────────────────────────
N_CLUSTERS   = 6
METRIC       = 'dtw'
MAX_ITER     = 100
RANDOM_STATE = 42

# Plotly colours for clusters 0–5
CLUSTER_COLOURS = [
    '#1f77b4', '#ff7f0e', '#2ca02c',
    '#d62728', '#9467bd', '#8c564b'
]

print('Paths ok')
print(f'NetCDF dir  : {NETCDF_DIR}')
print(f'Vector dir  : {VECTOR_DIR}')
print(f'Charts dir  : {CHARTS_DIR}')
print(f'Shapefile   : {SHAPEFILE}')

Paths ok
NetCDF dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\NetCDF_cubes
Vector dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_netcdf_vector
Charts dir  : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\results\metrics\charts_html
Shapefile   : C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\raw\GIS_lag\clusters_hovedstad_clean.shp


In [30]:
# ── Cell 2: Opdag NetCDF-filer og gruppér per variabel ─────────────────────────
# Filnavnsformat: cluster_{var}_long_{start}_{end}.nc
PATTERN = re.compile(r'^cluster_(.+)_long_(\d{4})_(\d{4})\.nc$')

def discover_variable_cubes(netcdf_dir: Path) -> dict:
    """Returnerer dict {variable: [(start, end, path), ...]}, sorteret efter interval."""
    grouped = defaultdict(list)
    for f in netcdf_dir.glob('*.nc'):
        m = PATTERN.match(f.name)
        if m:
            var, start, end = m.group(1), int(m.group(2)), int(m.group(3))
            grouped[var].append((start, end, f))
    # Sort each variable's list by interval start
    return {var: sorted(paths, key=lambda x: x[0]) for var, paths in sorted(grouped.items())}

VARIABLE_CUBES = discover_variable_cubes(NETCDF_DIR)

print(f'Fundet {len(VARIABLE_CUBES)} variable:')
for var, intervals in VARIABLE_CUBES.items():
    print(f'  {var:25s}  {len(intervals)} intervaller '
          f'({intervals[0][0]}–{intervals[-1][1]})')

Fundet 23 variable:
  EMUB                       6 intervaller (1990–2020)
  PMB                        6 intervaller (1990–2020)
  PUB                        6 intervaller (1990–2020)
  age_18_25                  6 intervaller (1990–2020)
  age_26_40                  6 intervaller (1990–2020)
  age_41_55                  6 intervaller (1990–2020)
  age_56_69                  6 intervaller (1990–2020)
  counts                     6 intervaller (1990–2020)
  crime_main_y               6 intervaller (1990–2020)
  disp_inc                   6 intervaller (1990–2020)
  emp                        6 intervaller (1990–2020)
  grund                      6 intervaller (1990–2020)
  gym_erhv                   6 intervaller (1990–2020)
  lvu                        6 intervaller (1990–2020)
  mean_price                 6 intervaller (1990–2020)
  mean_sqm                   6 intervaller (1990–2020)
  mig_in                     6 intervaller (1990–2020)
  mig_net                    6 intervaller (1

In [31]:
# ── Cell 3: Hjælpefunktioner: Indlæs & sammenkæd NetCDF kuber ─────────────────

def load_and_concat_variable(var: str, intervals: list) -> tuple:
    """
    Indlæser alle NetCDF-kuber for én variabel og sammenkæder dem langs tidsaksen.
    
    Returnerer:
        location_ids  – array af cluster-IDs (str)
        time_labels   – liste af tids-labels (str)
        X             – numpy array, shape (n_locations, n_timepoints)
    """
    arrays = []
    all_time_labels = []
    location_sets = []

    for start, end, path in intervals:
        ds = xr.open_dataset(path)
        # Find den primære DataArray (ikke koordinat-variable)
        data_vars = [v for v in ds.data_vars]
        da = ds[data_vars[0]] if data_vars else ds[var]

        # Forventet dims: (location, time)
        if 'location' not in da.dims or 'time' not in da.dims:
            # Prøv generisk: antag første dim = location, anden = time
            da = da.rename({da.dims[0]: 'location', da.dims[1]: 'time'})

        loc_ids = da.coords['location'].values.astype(str)
        time_coords = da.coords['time'].values

        # Tidslabel: brug interval-streng hvis koordinater ikke er sigende
        if len(time_coords) > 0:
            try:
                labels = [str(t)[:10] for t in time_coords]
            except Exception:
                labels = [f'{start}-{end}_t{i}' for i in range(len(time_coords))]
        else:
            labels = [f'{start}-{end}']

        arrays.append((loc_ids, da.values))  # values: (n_loc, n_time)
        all_time_labels.extend(labels)
        location_sets.append(set(loc_ids))
        ds.close()

    # Fælles locations (inner join) for at undgå NaN-rækker fra manglende kuber
    common_locs = location_sets[0]
    for s in location_sets[1:]:
        common_locs = common_locs & s
    common_locs = sorted(common_locs)

    # Sammensæt matrix
    concat_parts = []
    for loc_ids, values in arrays:
        loc_idx = {l: i for i, l in enumerate(loc_ids)}
        rows = [loc_idx[l] for l in common_locs]
        concat_parts.append(values[rows, :])

    X = np.concatenate(concat_parts, axis=1)  # (n_common_locs, total_timepoints)
    return np.array(common_locs), all_time_labels, X


# Hurtig smoke-test på én variabel
_var_test = list(VARIABLE_CUBES.keys())[0]
_locs, _tlabels, _X = load_and_concat_variable(_var_test, VARIABLE_CUBES[_var_test])
print(f'Test variabel  : {_var_test}')
print(f'Locations      : {len(_locs)}')
print(f'Tidspunkter    : {len(_tlabels)}')
print(f'Matrix shape   : {_X.shape}')
print(f'NaN-andel      : {np.isnan(_X).mean():.2%}')

Test variabel  : EMUB
Locations      : 1421
Tidspunkter    : 36
Matrix shape   : (1421, 36)
NaN-andel      : 0.00%


In [32]:
# ── Cell 4: TSC clustering-funktion ───────────────────────────────────────────

def run_tsc(X: np.ndarray, location_ids: np.ndarray) -> tuple:
    """
    Kører TimeSeriesKMeans (DTW) på matrix X (n_loc × n_time).
    
    Returnerer:
        labels  – cluster-ID per lokation (int array 0–5)
        km      – fitted TimeSeriesKMeans objekt
        X_scaled – skaleret data (til silhouette-beregning)
    """
    # NaN-håndtering: ffill → bfill → 0
    X_filled = np.zeros_like(X, dtype=float)
    for i in range(X.shape[0]):
        ts = pd.Series(X[i])
        filled = ts.ffill().bfill().fillna(0).values
        X_filled[i] = filled

    # Reshape til tslearn format: (n_samples, n_timesteps, 1)
    X_3d = X_filled[:, :, np.newaxis]

    # Standardisering
    scaler = TimeSeriesScalerMeanVariance()
    X_scaled = scaler.fit_transform(X_3d)

    # Clustering
    km = TimeSeriesKMeans(
        n_clusters=N_CLUSTERS,
        metric=METRIC,
        max_iter=MAX_ITER,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=0
    )
    labels = km.fit_predict(X_scaled)
    return labels, km, X_scaled


print('run_tsc() defineret.')

run_tsc() defineret.


In [33]:
# ── Cell 5: GeoPackage-eksportfunktion ─────────────────────────────────────────

def _detect_join_key(gdf: gpd.GeoDataFrame) -> str:
    """Find join-kolonne i shapefile (munic_clus → cluster_id → heuristik)."""
    for candidate in ['munic_clus', 'cluster_id', 'Cluster_id', 'CLUSTER_ID', 'cluster_ID']:
        if candidate in gdf.columns:
            return candidate
    # Heuristisk: kolonnenavn indeholder 'munic' eller 'clus'
    for col in gdf.columns:
        if 'munic' in col.lower() or 'clus' in col.lower():
            return col
    raise ValueError(f'Ingen passende join-kolonne fundet. Kolonner: {list(gdf.columns)}')


def save_gpkg(location_ids: np.ndarray, labels: np.ndarray, var: str,
              period_start: int = None, period_end: int = None) -> Path:
    """
    Merger cluster-labels til shapefile og gemmer som GeoPackage.
    Filnavn inkluderer periode-suffix hvis angivet.

    Returnerer output-stien.
    """
    gdf = gpd.read_file(SHAPEFILE)
    join_col = _detect_join_key(gdf)

    # Normalisér IDs: strip whitespace og leading zeros for konsistens
    gdf[join_col] = gdf[join_col].astype(str).str.strip().str.lstrip('0')

    cluster_df = pd.DataFrame({
        'location': pd.Series(location_ids).astype(str).str.strip().str.lstrip('0'),
        'cluster_tsc': labels.astype(int)
    })

    gdf_merged = gdf.merge(
        cluster_df,
        left_on=join_col,
        right_on='location',
        how='left'
    )

    matched = gdf_merged['cluster_tsc'].notna().sum()
    total   = len(gdf_merged)
    print(f'  Merge: {matched}/{total} lokationer matchet')

    period_str = f'{period_start}_{period_end}' if period_start is not None else 'full'
    out_path = VECTOR_DIR / f'{var}_{period_str}_tsc.gpkg'
    gdf_merged.to_file(out_path, driver='GPKG')
    return out_path


print('save_gpkg() defineret.')


save_gpkg() defineret.


In [34]:
# ── Cell 6: Plotly-visualiseringsfunktion ──────────────────────────────────────

def plot_centroids(km: TimeSeriesKMeans, time_labels: list, var: str,
                   period_start: int = None, period_end: int = None) -> Path:
    """
    Plotter 6 cluster-centroider som interaktive Plotly-linjer og gemmer HTML.
    Filnavn og titel inkluderer periode-suffix hvis angivet.

    Centroids shape: (n_clusters, n_timesteps, 1) → squeeze til (n_clusters, n_timesteps)
    """
    centroids = km.cluster_centers_[:, :, 0]  # (6, n_time)
    n_time = centroids.shape[1]

    # Brug kortere time_labels hvis de er for lange
    if len(time_labels) == n_time:
        x_labels = time_labels
    else:
        x_labels = list(range(n_time))

    period_str   = f'{period_start}_{period_end}' if period_start is not None else 'full'
    title_period = f' ({period_start}–{period_end})' if period_start is not None else ''

    fig = go.Figure()
    for c in range(N_CLUSTERS):
        fig.add_trace(go.Scatter(
            x=x_labels,
            y=centroids[c],
            mode='lines+markers',
            name=f'Cluster {c}',
            line=dict(color=CLUSTER_COLOURS[c], width=2),
            marker=dict(size=5)
        ))

    fig.update_layout(
        title=dict(
            text=f'TSC Centroider – {var}{title_period}',
            font=dict(size=18)
        ),
        xaxis_title='Tidspunkt',
        yaxis_title='Normaliseret værdi (z-score)',
        legend_title='Cluster',
        hovermode='x unified',
        template='plotly_white',
        height=500
    )

    out_path = CHARTS_DIR / f'{var}_{period_str}_tsc_timeseries.html'
    fig.write_html(str(out_path))
    return out_path


print('plot_centroids() defineret.')


plot_centroids() defineret.


In [35]:
# ── Cell 7: Summary-funktion ───────────────────────────────────────────────────

def compute_summary(var: str, labels: np.ndarray, km: TimeSeriesKMeans,
                    X_scaled: np.ndarray, n_intervals: int, n_timepoints: int) -> dict:
    """
    Beregner opsummerende statistik for én TSC-kørsel.
    Silhouette-score beregnes med Euclidean på fladtrykt matrix (hurtig approksimation).
    """
    cluster_sizes = pd.Series(labels).value_counts().sort_index()
    sizes_str = ', '.join(f'C{k}={v}' for k, v in cluster_sizes.items())

    # Silhouette (Euclidean approx på flattened 2D)
    X_flat = X_scaled[:, :, 0]  # (n_loc, n_time)
    try:
        sil = silhouette_score(X_flat, labels, metric='euclidean')
    except Exception:
        sil = np.nan

    return {
        'variable':     var,
        'n_locations':  len(labels),
        'n_intervals':  n_intervals,
        'n_timepoints': n_timepoints,
        'inertia':      round(km.inertia_, 4),
        'silhouette':   round(sil, 4) if not np.isnan(sil) else None,
        'cluster_sizes': sizes_str
    }


print('compute_summary() defineret.')

compute_summary() defineret.


In [36]:
# ── Cell 8: Hoved-batch loop ───────────────────────────────────────────────────
# Kører TSC på hvert interval separat for alle 23 variable (23 × 6 = 138 kørsler).

summary_rows = []
failed_vars  = []

total_runs = sum(len(ivs) for ivs in VARIABLE_CUBES.values())
run_num = 0

for var, intervals in VARIABLE_CUBES.items():
    for (start, end, path) in intervals:
        run_num += 1
        print(f'\n[{run_num}/{total_runs}] {var}  {start}–{end} ...')

        try:
            # 1. Indlæs ét interval
            location_ids, time_labels, X = load_and_concat_variable(var, [(start, end, path)])
            print(f'  Indlæst  : {X.shape[0]} lokationer × {X.shape[1]} tidspunkter')

            # 2. Kør TSC
            labels, km, X_scaled = run_tsc(X, location_ids)
            unique, counts = np.unique(labels, return_counts=True)
            print(f'  Clustered: {dict(zip(unique.tolist(), counts.tolist()))}')

            # 3. Gem GeoPackage
            gpkg_path = save_gpkg(location_ids, labels, var, start, end)
            print(f'  Gemt gpkg: {gpkg_path.name}')

            # 4. Plot og gem chart
            chart_path = plot_centroids(km, time_labels, var, start, end)
            print(f'  Gemt html: {chart_path.name}')

            # 5. Summary
            row = compute_summary(var, labels, km, X_scaled, 1, X.shape[1])
            row['period_start'] = start
            row['period_end']   = end
            summary_rows.append(row)
            print(f'  Silhouette={row["silhouette"]}, Inertia={row["inertia"]}')

        except Exception as e:
            print(f'  FEJL: {e}')
            failed_vars.append((f'{var} {start}-{end}', str(e)))

print(f'\n{"="*60}')
print(f'Færdig: {len(summary_rows)}/{total_runs} kørsler gennemført.')
if failed_vars:
    print('Fejlede:')
    for v, err in failed_vars:
        print(f'  {v}: {err}')



[1/138] EMUB  1990–1995 ...
  Indlæst  : 1421 lokationer × 6 tidspunkter
  Clustered: {0: 178, 1: 284, 2: 301, 3: 215, 4: 229, 5: 214}
  Merge: 1421/1421 lokationer matchet
  Gemt gpkg: EMUB_1990_1995_tsc.gpkg
  Gemt html: EMUB_1990_1995_tsc_timeseries.html
  Silhouette=0.1121, Inertia=1.6312

[2/138] EMUB  1995–2000 ...
  Indlæst  : 1421 lokationer × 6 tidspunkter
  Clustered: {0: 247, 1: 247, 2: 176, 3: 232, 4: 291, 5: 228}
  Merge: 1421/1421 lokationer matchet
  Gemt gpkg: EMUB_1995_2000_tsc.gpkg
  Gemt html: EMUB_1995_2000_tsc_timeseries.html
  Silhouette=0.0984, Inertia=1.6584

[3/138] EMUB  2000–2005 ...
  Indlæst  : 1421 lokationer × 6 tidspunkter
  Clustered: {0: 243, 1: 280, 2: 334, 3: 159, 4: 191, 5: 214}
  Merge: 1421/1421 lokationer matchet
  Gemt gpkg: EMUB_2000_2005_tsc.gpkg
  Gemt html: EMUB_2000_2005_tsc_timeseries.html
  Silhouette=0.1138, Inertia=1.6691

[4/138] EMUB  2005–2010 ...
  Indlæst  : 1421 lokationer × 6 tidspunkter
  Clustered: {0: 315, 1: 261, 2: 215, 3: 

In [37]:
# ── Cell 9: Gem summary CSV og vis resultat ────────────────────────────────────

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

print(f'Summary gemt: {SUMMARY_CSV}')
print(f'\nFiler i tsc_netcdf_vector/:')
gpkg_files = sorted(VECTOR_DIR.glob('*.gpkg'))
for f in gpkg_files:
    print(f'  {f.name}')

print(f'\nFiler i charts_html/ (tsc):')
chart_files = sorted(CHARTS_DIR.glob('*_tsc_timeseries.html'))
for f in chart_files:
    print(f'  {f.name}')

print(f'\nSummary tabel:')
cols = ['variable', 'period_start', 'period_end', 'n_locations',
        'n_timepoints', 'inertia', 'silhouette', 'cluster_sizes']
display(summary_df[[c for c in cols if c in summary_df.columns]])


Summary gemt: C:\Users\jonas\Desktop\7_semester\Projekt\Gentrification_model\data\processed\tsc_netcdf_summary.csv

Filer i tsc_netcdf_vector/:
  age_18_25_1990_1995_tsc.gpkg
  age_18_25_1995_2000_tsc.gpkg
  age_18_25_2000_2005_tsc.gpkg
  age_18_25_2005_2010_tsc.gpkg
  age_18_25_2010_2015_tsc.gpkg
  age_18_25_2015_2020_tsc.gpkg
  age_18_25_tsc.gpkg
  age_26_40_1990_1995_tsc.gpkg
  age_26_40_1995_2000_tsc.gpkg
  age_26_40_2000_2005_tsc.gpkg
  age_26_40_2005_2010_tsc.gpkg
  age_26_40_2010_2015_tsc.gpkg
  age_26_40_2015_2020_tsc.gpkg
  age_26_40_tsc.gpkg
  age_41_55_1990_1995_tsc.gpkg
  age_41_55_1995_2000_tsc.gpkg
  age_41_55_2000_2005_tsc.gpkg
  age_41_55_2005_2010_tsc.gpkg
  age_41_55_2010_2015_tsc.gpkg
  age_41_55_2015_2020_tsc.gpkg
  age_41_55_tsc.gpkg
  age_56_69_1990_1995_tsc.gpkg
  age_56_69_1995_2000_tsc.gpkg
  age_56_69_2000_2005_tsc.gpkg
  age_56_69_2005_2010_tsc.gpkg
  age_56_69_2010_2015_tsc.gpkg
  age_56_69_2015_2020_tsc.gpkg
  age_56_69_tsc.gpkg
  counts_1990_1995_tsc.gpkg


,variable,period_start,period_end,n_locations,n_timepoints,inertia,silhouette,cluster_sizes
0,EMUB,1990,1995,1421,6,1.6312,0.1121,"C0=178, C1=284, C2=301, C3=215, C4=229, C5=214"
1,EMUB,1995,2000,1421,6,1.6584,0.0984,"C0=247, C1=247, C2=176, C3=232, C4=291, C5=228"
2,EMUB,2000,2005,1421,6,1.6691,0.1138,"C0=243, C1=280, C2=334, C3=159, C4=191, C5=214"
3,EMUB,2005,2010,1421,6,1.6366,0.1142,"C0=315, C1=261, C2=215, C3=208, C4=191, C5=231"
4,EMUB,2010,2015,1421,6,1.6222,0.1380,"C0=211, C1=360, C2=262, C3=169, C4=194, C5=225"
...,...,...,...,...,...,...,...,...
133,unemp,1995,2000,1421,6,0.9293,0.1081,"C0=114, C1=578, C2=55, C3=311, C4=244, C5=119"
134,unemp,2000,2005,1421,6,1.7185,0.0465,"C0=227, C1=388, C2=208, C3=209, C4=212, C5=177"
135,unemp,2005,2010,1421,6,1.5042,0.0498,"C0=194, C1=302, C2=261, C3=161, C4=291, C5=212"
136,unemp,2010,2015,1421,6,1.6171,0.0681,"C0=197, C1=278, C2=220, C3=320, C4=167, C5=239"


In [38]:
# ── Cell 10 (valgfri): Vis én chart inline for visuel verifikation ─────────────

PREVIEW_VAR = 'disp_inc'  # Skift til en anden variabel efter ønske

if PREVIEW_VAR in VARIABLE_CUBES:
    _locs, _tlabels, _X = load_and_concat_variable(PREVIEW_VAR, VARIABLE_CUBES[PREVIEW_VAR])
    _labels, _km, _X_scaled = run_tsc(_X, _locs)
    _centroids = _km.cluster_centers_[:, :, 0]

    fig_preview = go.Figure()
    for c in range(N_CLUSTERS):
        n_in_cluster = int((_labels == c).sum())
        fig_preview.add_trace(go.Scatter(
            x=list(range(len(_tlabels))),
            y=_centroids[c],
            mode='lines+markers',
            name=f'Cluster {c} (n={n_in_cluster})',
            line=dict(color=CLUSTER_COLOURS[c], width=2),
            marker=dict(size=5)
        ))
    fig_preview.update_layout(
        title=f'Preview: TSC centroider – {PREVIEW_VAR}',
        xaxis_title='Tidspunkt (indeks)',
        yaxis_title='Normaliseret z-score',
        template='plotly_white',
        height=500
    )
    fig_preview.show()
else:
    print(f'{PREVIEW_VAR} ikke fundet i VARIABLE_CUBES.')